<a href="https://colab.research.google.com/github/PollyIva/DL_in_CV/blob/PollyIva-patch-1/DeepSORT_project_3" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Установка библиотек и Импорт



In [3]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [5]:
!pip install ultralytics
!pip install torchreid
!pip install motmetrics
!pip install gdown
!pip install onnxruntime --quiet

!git clone https://github.com/nwojke/deep_sort.git
%cd deep_sort
!pip install -r requirements.txt
%cd ..

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=70392440370e9b0e654f346fa60582213dc28b760afae1faab43ff44d046e8da
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 62.9 MB/s eta 0:00:00
Cloning into 'deep_sort'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 167 (delta 4), 

In [6]:
import os
import sys
sys.path.append("deep_sort")
print("DeepSORT path added")
import cv2
import torch
import numpy as np
import time

import torchreid
import torchvision.transforms as T
from PIL import Image
import csv
import json
import hashlib
sys.path.append('./deep_sort')

from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker
from ultralytics import YOLO
import onnxruntime as ort

import pandas as pd
import shutil
import tempfile
# import kagglehub
!git clone https://github.com/JonathonLuiten/TrackEval.git

import sys
sys.path.append("TrackEval")

import trackeval
print(trackeval.__file__)

from trackeval import Evaluator
from trackeval.datasets import MotChallenge2DBox
from trackeval.metrics import HOTA, CLEAR, Identity

DeepSORT path added


/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 340), reused 315 (delta 315), pack-reused 549 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 8.95 MiB/s, done.
Resolving deltas: 100% (619/619), done.
/content/TrackEval/trackeval/__init__.py


In [7]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


# 2. Настройки проекта

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
BASE_DIR = "/content/drive/MyDrive/DeepSORT"

folders = [
    "videos",
    "results",
    "overlays",
    "metrics",
    "configs"
]

# Создаем основную папку
os.makedirs(BASE_DIR, exist_ok=True)

# Создаем подпапки
for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Folders created")

Folders created


In [12]:
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
OVERLAYS_DIR = os.path.join(BASE_DIR, "overlays")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")
CONFIGS_DIR = os.path.join(BASE_DIR, "configs")

В диск были скопированы датасеты
https://www.kaggle.com/datasets/takshmandar/mot16-dataset
https://www.kaggle.com/datasets/mdraselsarker/mot15-challenge-dataset?resource=download

In [13]:
DATA_DIR = os.path.join(BASE_DIR, "MOT_datasets")
os.makedirs(DATA_DIR, exist_ok=True)

In [15]:
SEQUENCES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(SEQUENCES)

['MOT_datasets', 'configs', 'metrics', 'overlays', 'results', 'videos']


In [33]:
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def make_key(*args):
    raw = "_".join(map(str, args))
    return hashlib.md5(raw.encode()).hexdigest()


def load_cache(key):
    path = os.path.join(CACHE_DIR, f"{key}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


def save_cache(key, data):
    path = os.path.join(CACHE_DIR, f"{key}.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

таблица результатов

In [34]:
os.makedirs("/content/drive/MyDrive/DeepSORT", exist_ok=True)

output_csv = os.path.join(BASE_DIR, "experiments.csv")
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

fieldnames = [
    "sequence", "detector", "reid",
    "conf", "cosine_distance", "nn_budget",
    "fps", "MOTA", "IDF1",
    "HOTA"
]

file_exists = os.path.exists(output_csv)

#3. Detectors

In [16]:
#Базовый класс детектора
class Detector:
    def detect(self, frame):
        raise NotImplementedError

In [17]:
class YOLOv5Detector(Detector):
    def __init__(self, model_name="yolov5n.pt", conf=0.5, device=DEVICE):
        self.model = YOLO(model_name)  # ultralytics wrapper
        self.conf = conf
        self.device = device

    def detect(self, frame):
        result = self.model(frame, conf=self.conf, device=self.device, verbose=False)[0]

        detections = []
        for box in result.boxes:
            if int(box.cls[0]) != 0:  # person class
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            detections.append([
                int(x1), int(y1), int(x2), int(y2), float(box.conf[0])
            ])

        return detections

In [18]:
class NanoDetDetector(Detector):
    def __init__(self, model_path="nanodet_tiny.onnx", conf=0.5, input_size=416):
        self.session = ort.InferenceSession(model_path)
        self.conf = conf
        self.input_size = input_size

    def detect(self, frame):
        img = cv2.resize(frame, (self.input_size, self.input_size))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))[None, ...]

        outputs = self.session.run(None, {"input": img})

        detections = []

        # TODO: распарсить outputs в формат [x1,y1,x2,y2,score]

        return detections

In [19]:
#  Фабрика детекторов

def create_detector(name, conf=0.5):
    if name == "yolov5n":
        return YOLOv5Detector("yolov5n.pt", conf=conf)
    elif name == "yolov5m":
        return YOLOv5Detector("yolov5m.pt", conf=conf)
    elif name == "nanodet_t":
        return NanoDetDetector("nanodet_tiny.onnx", conf=conf, input_size=416)
    else:
        raise ValueError(f"Unknown detector: {name}")

In [20]:
detector_name = "yolov5n"
detector = create_detector(detector_name)
print(
    "Loaded detector:",
    detector_name
)

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Loaded detector: yolov5n


# 4. ReID


In [21]:
class ReIDModel:
    def get_embedding(self, image):
        raise NotImplementedError

In [22]:
class TorchReID(ReIDModel):

    def __init__(
        self,
        model_name,
        device=DEVICE
    ):
        self.device = device

        self.model = torchreid.models.build_model(
            name=model_name,
            num_classes=1000,
            pretrained=True
        )

        self.model.to(device)
        self.model.eval()

        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    @torch.no_grad()
    def get_embedding(self, image):

        img = self.transform(image)

        img = img.unsqueeze(0).to(self.device)

        feature = self.model(img)

        feature = feature.cpu().numpy()[0]

        # L2 normalization
        feature /= np.linalg.norm(feature)

        return feature

In [23]:
def create_reid(name):

    if name == "osnet_x0_25":
        return TorchReID(
            "osnet_x0_25"
        )

    elif name == "osnet_x1_0":
        return TorchReID(
            "osnet_x1_0"
        )

    elif name == "resnet50":
        return TorchReID(
            "resnet50"
        )

    else:
        raise ValueError(
            f"Unknown REID model: {name}"
        )

In [24]:
reid_name = "osnet_x0_25"

reid = create_reid(reid_name)

print(
    "Loaded REID:",
    reid_name
)

Downloading...
From: https://drive.google.com/uc?id=1rb8UN5ZzPKRc_xvtHlyDh-cSz88YX9hs
To: /root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth
100%|██████████| 2.97M/2.97M [00:00<00:00, 197MB/s]

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Loaded REID: osnet_x0_25


# 5. DeepSORT.

In [25]:
class DeepSORTWrapper:

    def __init__(
        self,
        max_cosine_distance=0.3,
        nn_budget=100
    ):

        self.metric = nn_matching.NearestNeighborDistanceMetric(
            "cosine",
            max_cosine_distance,
            nn_budget
        )

        self.tracker = Tracker(self.metric)
    def update(
        self,
        frame,
        detections,
        reid
    ):
        """
        detections:
        [x1, y1, x2, y2, confidence]
        """

        ds_detections = []


        for det in detections:

            x1, y1, x2, y2, conf = det


            # Вырезаем человека
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0: continue

            feature = reid.get_embedding(crop)


            # DeepSORT использует tlwh
            bbox = [
                x1,
                y1,
                x2 - x1,
                y2 - y1
            ]


            ds_detections.append(
                Detection(
                    bbox,
                    conf,
                    feature
                )
            )
        # Предсказание нового положения треков
        self.tracker.predict()


        # Сопоставление детекций и треков
        self.tracker.update(
            ds_detections
        )


        results = []


        for track in self.tracker.tracks:
            if not track.is_confirmed():
                continue

            if track.time_since_update > 1:
                continue

            bbox = track.to_tlbr()
            results.append({
                "id": track.track_id,
                "bbox": bbox
            })


        return results

In [26]:
tracker = DeepSORTWrapper(
    max_cosine_distance=0.3,
    nn_budget=100
)
detector = create_detector("yolov5n")
reid = create_reid("osnet_x0_25")

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


# 6. сохранение результатов и Визуализация



In [27]:
class MOTWriter:
    def __init__(self, filename):
        self.file = open(filename, "w")


    def write(self, frame_id, track_id, bbox):
        x1, y1, x2, y2 = bbox

        width = x2 - x1
        height = y2 - y1

        line = (
            f"{frame_id},{track_id},"
            f"{x1:.2f},{y1:.2f},"
            f"{width:.2f},{height:.2f},"
            "1,-1,-1,-1\n"
        )

        self.file.write(line)


    def close(self):
        self.file.close()

In [28]:
def draw_tracks(frame, tracks):

    for track in tracks:

        x1, y1, x2, y2 = map(int, track["bbox"])

        track_id = track["id"]

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    return frame

In [29]:
def run_sequence(
    img_dir,
    output_video,
    mot_path,
    detector,
    reid,
    tracker,
    fps=30
):

    frames = sorted(os.listdir(img_dir))

    first_frame = cv2.imread(os.path.join(img_dir, frames[0]))
    h, w = first_frame.shape[:2]

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    mot_writer = MOTWriter(mot_path)

    total_time = 0

    for i, frame_name in enumerate(frames, start=1):

        frame_path = os.path.join(img_dir, frame_name)
        frame = cv2.imread(frame_path)

        if frame is None:
            continue

        start = time.time()

        detections = detector.detect(frame)
        tracks = tracker.update(frame, detections, reid)

        elapsed = time.time() - start
        total_time += elapsed

        fps_now = 1 / elapsed if elapsed > 0 else 0

        cv2.putText(frame, f"FPS: {fps_now:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        frame = draw_tracks(frame, tracks)

        for t in tracks:
            mot_writer.write(i, t["id"], t["bbox"])

        writer.write(frame)

        if i % 50 == 0:
            print(f"Frame {i}/{len(frames)} FPS={fps_now:.2f}")

    writer.release()
    mot_writer.close()

    avg_fps = len(frames) / total_time if total_time > 0 else 0

    print("DONE")
    print("AVG FPS:", avg_fps)

    return avg_fps

10. Grid Search

11. Анализ результатов

In [31]:
detectors = [
    "yolov5n",
    "yolov5m",
    # "nanodet_t",
]


reids = [
    "osnet_x0_25",
    # "osnet_x1_0",
    # "resnet50"
]


confidences = [
    # 0.3,
    0.5,
    # 0.7
]


cosine_distances = [
#     0.2,
#     0.3,
    0.5
]


nn_budgets = [
    50,
    # 100
]

In [35]:
def compute_metrics(sequence_path, pred_file):
    seq_name = os.path.basename(sequence_path)

    if "MOT16" in sequence_path:
        benchmark = "MOT16"
    else:
        benchmark = "MOT15"

    tmp = tempfile.mkdtemp()
    gt_root = os.path.join(tmp, "gt")
    tracker_root = os.path.join(tmp, "trackers")

    try:
        # Путь к папке последовательности ОТНОСИТЕЛЬНО GT_FOLDER
        seq_rel_path = os.path.join(benchmark, "train", seq_name)
        gt_seq_dir = os.path.join(gt_root, seq_rel_path)
        os.makedirs(gt_seq_dir, exist_ok=True)

        # Копируем seqinfo.ini
        src_ini = os.path.join(sequence_path, "seqinfo.ini")
        dst_ini = os.path.join(gt_seq_dir, "seqinfo.ini")
        shutil.copy(src_ini, dst_ini)

        # Создаём папку gt и копируем туда gt.txt
        gt_gt_dir = os.path.join(gt_seq_dir, "gt")
        os.makedirs(gt_gt_dir, exist_ok=True)
        shutil.copy(
            os.path.join(sequence_path, "gt", "gt.txt"),
            os.path.join(gt_gt_dir, "gt.txt")
        )

        # Результаты трекера
        tracker_name = "DeepSORT"
        tracker_dir = os.path.join(tracker_root, benchmark, "train", tracker_name, "data")
        os.makedirs(tracker_dir, exist_ok=True)
        shutil.copy(pred_file, os.path.join(tracker_dir, f"{seq_name}.txt"))

        # Seqmap: сначала определяем путь, потом пишем файл
        seqmap_dir = os.path.join(tmp, "seqmaps")
        os.makedirs(seqmap_dir, exist_ok=True)

        # ВАЖНО: объявляем переменную seqmap_path ДО того, как использовать её
        seqmap_path = os.path.join(seqmap_dir, f"{benchmark}-train.txt")

        with open(seqmap_path, "w") as f:
            f.write("name\n")
            f.write(seq_rel_path + "\n")  # MOT15/train/TUD-Campus

        # --- ОТЛАДКА (теперь безопасно, потому что seqmap_path уже создан) ---
        print("=== DEBUG: contents of GT_FOLDER ===")
        for root, dirs, files in os.walk(gt_root):
            level = root.replace(gt_root, '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for f in files:
                print(f"{subindent}{f}")

        print("=== DEBUG: seqmap contents ===")
        with open(seqmap_path) as f:
            print(f.read())
        # --------------------------------------------------------------------

        eval_config = Evaluator.get_default_eval_config()
        dataset_config = MotChallenge2DBox.get_default_dataset_config()

        dataset_config["GT_FOLDER"] = gt_root
        dataset_config["TRACKERS_FOLDER"] = tracker_root
        dataset_config["BENCHMARK"] = benchmark
        dataset_config["SPLIT_TO_EVAL"] = "train"
        dataset_config["SEQMAP_FOLDER"] = seqmap_dir
        dataset_config["TRACKERS_TO_EVAL"] = [tracker_name]

        evaluator = Evaluator(eval_config)
        dataset = MotChallenge2DBox(dataset_config)

        metrics = [HOTA(), CLEAR(), Identity()]
        results, _ = evaluator.evaluate([dataset], metrics)

        res = results["MotChallenge2DBox"][tracker_name]["COMBINED_SEQ"]
        return {
            "MOTA": float(res["CLEAR"]["MOTA"]),
            "IDF1": float(res["Identity"]["IDF1"]),
            "HOTA": float(res["HOTA"]["HOTA"]),
        }

    finally:
        shutil.rmtree(tmp)


In [36]:
def run_experiment(video_path, sequence,
                   detector_name, reid_name,
                   conf, cosine_distance, nn_budget):

    os.makedirs(RESULTS_DIR, exist_ok=True)

    sequence_name = os.path.basename(sequence)

    detector = create_detector(detector_name)
    detector.conf = conf

    reid = create_reid(reid_name)

    tracker = DeepSORTWrapper(
        max_cosine_distance=cosine_distance,
        nn_budget=nn_budget
    )

    output_video = os.path.join(
        "results",
        f"{sequence_name}_{detector_name}_{reid_name}.mp4"
    )

    output_mot = os.path.join(
        "results",
        f"{sequence_name}_{detector_name}_{reid_name}.txt"
    )

    fps = run_sequence(
        img_dir=video_path,
        output_video=output_video,
        mot_path=output_mot,
        detector=detector,
        reid=reid,
        tracker=tracker
    )

    return {
        "fps": fps,
        "mot_file": output_mot
    }

In [37]:
def run_cached(sequence, video_path,
               detector, reid,
               conf, cosine, budget):

    key = make_key(sequence, detector, reid, conf, cosine, budget)

    cached = load_cache(key)
    if cached:
        print("CACHE HIT:", sequence, detector, reid)
        return cached

    print("RUN:", sequence, detector, reid)

    result = run_experiment(
        video_path, sequence,
        detector, reid,
        conf, cosine, budget
    )

    save_cache(key, result)
    return result

In [38]:
with open(output_csv, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    if not file_exists:
        writer.writeheader()

    for sequence in SEQUENCES:

        video_path = os.path.join(sequence,  "img1")

        for detector in detectors:
            for reid in reids:
                for conf in confidences:
                    for cosine in cosine_distances:
                        for budget in nn_budgets:

                            try:
                                exp = run_cached(
                                    sequence,
                                    video_path,
                                    detector,
                                    reid,
                                    conf,
                                    cosine,
                                    budget
                                )

                                metrics = compute_metrics(sequence, exp["mot_file"])

                                row = {
                                    "sequence": sequence,
                                    "detector": detector,
                                    "reid": reid,
                                    "conf": conf,
                                    "cosine_distance": cosine,
                                    "nn_budget": budget,
                                    "fps": exp["fps"],
                                    **metrics
                                }

                                writer.writerow(row)
                                f.flush()

                                print("OK:", sequence, detector, reid)

                            except Exception as e:
                                print("FAILED:", sequence, detector, reid)
                                print(e)

RUN: MOT_datasets yolov5n osnet_x0_25
PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
FAILED: MOT_datasets yolov5n osnet_x0_25
[Errno 2] No such file or directory: 'MOT_datasets/img1'
RUN: MOT_datasets yolov5m osnet_x0_25
PRO TIP 💡 Replace 'model=yolov5m.pt' with new 'model=yolov5mu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
FAILED: MOT_datasets yolov5m osnet_x0_25
[Errno 2] No such file or directory: 'MOT_datasets/

In [ ]:
with open(output_csv, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    if not file_exists:
        writer.writeheader()

    for sequence_path, seq_info in SEQUENCES.items():

        video_path = os.path.join(sequence_path, "img1")

        for detector in detectors:
            for reid in reids:
                for conf in confidences:
                    for cosine in cosine_distances:
                        for budget in nn_budgets:

                            try:
                                exp = run_cached(
                                    sequence_path,
                                    video_path,
                                    detector,
                                    reid,
                                    conf,
                                    cosine,
                                    budget
                                )

                                metrics = compute_metrics(
                                    sequence_path,
                                    exp["mot_file"]
                                )

                                row = {
                                    "sequence": seq_info["sequence"],
                                    "benchmark": seq_info["benchmark"],
                                    "detector": detector,
                                    "reid": reid,
                                    "conf": conf,
                                    "cosine_distance": cosine,
                                    "nn_budget": budget,
                                    "fps": exp["fps"],
                                    **metrics
                                }

                                writer.writerow(row)
                                f.flush()

                                print(
                                    f"OK: {seq_info['sequence']} "
                                    f"{detector} {reid}"
                                )

                            except Exception as e:
                                print(
                                    f"FAILED: {seq_info['sequence']} "
                                    f"{detector} {reid}"
                                )
                                traceback.print_exc()